Cell 1: Imports and System Path Configuration

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path so we can import from src
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display

from src.dsp.filters import load_and_preprocess_audio, compute_shannon_energy_envelope

print("Environment configured successfully.")

Cell 2: Parse Ground Truth Clinical Labels

In [ ]:
data_dir = project_root / "data" / "raw" / "training-a"
ref_file = data_dir / "REFERENCE.csv"

# Load references (file_id, label)
df_ref = pd.read_csv(ref_file, header=None, names=["record_id", "label"])
df_ref["class"] = df_ref["label"].map({1: "Normal (Laminar)", -1: "Abnormal (Turbulent)"})

print(f"Total recordings in training-a: {len(df_ref)}")
print(df_ref["class"].value_counts())
df_ref.head()

Cell 3: Load One Normal and One Abnormal Recording

In [ ]:
# Grab first normal and first abnormal sample
normal_id = df_ref[df_ref["label"] == 1].iloc[0]["record_id"]
abnormal_id = df_ref[df_ref["label"] == -1].iloc[0]["record_id"]

normal_path = str(data_dir / f"{normal_id}.wav")
abnormal_path = str(data_dir / f"{abnormal_id}.wav")

print(f"Loading Normal: {normal_id}.wav")
raw_norm, filt_norm, fs = load_and_preprocess_audio(normal_path, target_sr=4000)

print(f"Loading Abnormal: {abnormal_id}.wav")
raw_abnorm, filt_abnorm, _ = load_and_preprocess_audio(abnormal_path, target_sr=4000)

# Compute Shannon Envelopes
env_norm = compute_shannon_energy_envelope(filt_norm, fs=fs)
env_abnorm = compute_shannon_energy_envelope(filt_abnorm, fs=fs)

Cell 4: Visualizing Raw vs. Filtered Waveforms & Shannon Envelopes
To inspect cardiac cycles cleanly, zoom into a 5-second window:

In [ ]:
duration = 5.0 # seconds
n_samples = int(duration * fs)
t = np.linspace(0, duration, n_samples)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

# 1. Normal Filtered Waveform
axes[0].plot(t, filt_norm[:n_samples], color="#1f77b4", lw=0.9)
axes[0].set_title(f"Normal Sample ({normal_id}) - 4th-Order Bandpass (50–1200 Hz)", fontweight="bold")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True, alpha=0.3)

# 2. Normal Shannon Energy Envelope
axes[1].plot(t, env_norm[:n_samples], color="#2ca02c", lw=1.2)
axes[1].fill_between(t, 0, env_norm[:n_samples], color="#2ca02c", alpha=0.2)
axes[1].set_title("Normal Sample - Normalized Shannon Energy Envelope", fontweight="bold")
axes[1].set_ylabel("Energy")
axes[1].grid(True, alpha=0.3)

# 3. Abnormal Filtered Waveform
axes[2].plot(t, filt_abnorm[:n_samples], color="#d62728", lw=0.9)
axes[2].set_title(f"Abnormal Sample ({abnormal_id}) - Filtered Waveform (Murmur / Turbulent Jet)", fontweight="bold")
axes[2].set_ylabel("Amplitude")
axes[2].grid(True, alpha=0.3)

# 4. Abnormal Shannon Energy Envelope
axes[3].plot(t, env_abnorm[:n_samples], color="#ff7f0e", lw=1.2)
axes[3].fill_between(t, 0, env_abnorm[:n_samples], color="#ff7f0e", alpha=0.2)
axes[3].set_title("Abnormal Sample - Normalized Shannon Energy Envelope", fontweight="bold")
axes[3].set_xlabel("Time (seconds)")
axes[3].set_ylabel("Energy")
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Cell 5: Time-Frequency Mel-Spectrogram & Spectral Centroid Comparison
This is the core diagnostic check:

In [ ]:
# Compute Mel-Spectrograms (fmax=1200 matches our bandpass)
n_mels = 64
n_fft = 512
hop_length = 128

S_norm = librosa.feature.melspectrogram(y=filt_norm[:n_samples], sr=fs, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, fmax=1200)
S_norm_db = librosa.power_to_db(S_norm, ref=np.max)

S_abnorm = librosa.feature.melspectrogram(y=filt_abnorm[:n_samples], sr=fs, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, fmax=1200)
S_abnorm_db = librosa.power_to_db(S_abnorm, ref=np.max)

# Compute Spectral Centroid
cent_norm = librosa.feature.spectral_centroid(y=filt_norm[:n_samples], sr=fs, n_fft=n_fft, hop_length=hop_length)[0]
cent_abnorm = librosa.feature.spectral_centroid(y=filt_abnorm[:n_samples], sr=fs, n_fft=n_fft, hop_length=hop_length)[0]

times = librosa.times_like(cent_norm, sr=fs, hop_length=hop_length)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Normal Spectrogram + Centroid
img1 = librosa.display.specshow(S_norm_db, sr=fs, hop_length=hop_length, x_axis='time', y_axis='mel', fmax=1200, ax=axes[0], cmap='magma')
axes[0].plot(times, cent_norm, color='cyan', lw=1.5, label='Spectral Centroid (Hz)')
axes[0].set_title(f"Normal Acoustic Spectrum ({normal_id}) — Low-frequency concentration", fontweight="bold")
axes[0].legend(loc="upper right")
fig.colorbar(img1, ax=axes[0], format='%+2.0f dB')

# Abnormal Spectrogram + Centroid
img2 = librosa.display.specshow(S_abnorm_db, sr=fs, hop_length=hop_length, x_axis='time', y_axis='mel', fmax=1200, ax=axes[1], cmap='magma')
axes[1].plot(times, cent_abnorm, color='cyan', lw=1.5, label='Spectral Centroid (Hz)')
axes[1].set_title(f"Abnormal Acoustic Spectrum ({abnormal_id}) — Turbulent Spectral Flare", fontweight="bold")
axes[1].legend(loc="upper right")
fig.colorbar(img2, ax=axes[1], format='%+2.0f dB')

plt.tight_layout()
plt.show()

print(f"Mean Spectral Centroid (Normal):   {cent_norm.mean():.1f} Hz")
print(f"Mean Spectral Centroid (Abnormal): {cent_abnorm.mean():.1f} Hz")